In [ ]:
!pip install -U pytubefix demucs

In [ ]:
!pip install -U pytubefix

In [ ]:

from pathlib import Path
import re
import time

from pytubefix import Search, YouTube


TRACKS = [
    #"КиШ — Лесник",
    #"Portishead — Glory Box",
    "Adele — Set Fire to the Rain",
    "Земфира — ПММЛ",
    "Muse — Feeling Good",
    "Fleetwood Mac — Dreams",
    "Avril Lavigne — Complicated",
    "Animal Джаz — Три полоски",
]

# Папка сохранения в Windows
OUTPUT_DIR = Path.home() / "Audio Stems" / "YouTube Audio"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def safe_filename(title: str) -> str:
    """Удаляет символы, запрещённые в именах файлов Windows."""
    title = re.sub(r'[<>:"/\\|?*]', "_", title)
    title = re.sub(r"\s+", " ", title).strip()
    title = title.rstrip(". ")

    # Защита от зарезервированных имён Windows
    if title.upper() in {
        "CON", "PRN", "AUX", "NUL",
        *(f"COM{i}" for i in range(1, 10)),
        *(f"LPT{i}" for i in range(1, 10)),
    }:
        title = "_" + title

    return title or "untitled"


def download_track(query: str) -> None:
    print(f"\nПоиск: {query}")

    try:
        results = Search(query).videos

        if not results:
            print("  Видео не найдено.")
            return

        # Первый результат поиска YouTube
        video_url = results[0].watch_url
        yt = YouTube(video_url)

        print(f"  Название: {yt.title}")
        print(f"  Ссылка: {video_url}")

        # Предпочитаем аудио AAC в контейнере MP4/M4A
        stream = yt.streams.get_audio_only(subtype="mp4")

        # Если такого потока нет, выбираем другой доступный
        if stream is None:
            stream = yt.streams.get_audio_only()

        if stream is None:
            print("  Аудиопоток не найден.")
            return

        filename = safe_filename(yt.title) + "." + stream.subtype
        destination = OUTPUT_DIR / filename

        if destination.exists():
            print("  Файл уже существует, пропускаю.")
            return

        stream.download(
            output_path=str(OUTPUT_DIR),
            filename=filename,
        )

        print(f"  Скачано: {destination}")

    except Exception as exc:
        print(f"  Ошибка: {type(exc).__name__}: {exc}")


def main() -> None:
    print(f"Папка сохранения: {OUTPUT_DIR}")
    print(f"Количество треков: {len(TRACKS)}")

    for index, track in enumerate(TRACKS, start=1):
        print(f"\n[{index}/{len(TRACKS)}]")
        download_track(track)

        # Пауза между запросами
        time.sleep(2)

    print("\nОбработка завершена.")


if __name__ == "__main__":
    main()


In [ ]:

from pathlib import Path
import re
import shutil
import subprocess
import sys
import time

from pytubefix import Search, YouTube


TRACKS = [
    "КиШ — Лесник",
    "Portishead — Glory Box",
    "Adele — Set Fire to the Rain",
    "Земфира — ПММЛ",
    "Muse — Feeling Good",
    "Fleetwood Mac — Dreams",
    "Avril Lavigne — Complicated",
    "Animal Джаz — Три полоски",
]

OUTPUT_DIR = Path.home() / "Audio Stems" / "YouTube Audio"
STEMS_DIR = OUTPUT_DIR / "Stems"
TEMP_DIR = OUTPUT_DIR / "_demucs_temp"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
STEMS_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

STEM_NAMES = [
    "vocals",
    "drums",
    "bass",
    "guitar",
    "piano",
    "other",
]


def safe_filename(title: str) -> str:
    """Создаёт безопасное имя файла или папки в Windows."""
    title = re.sub(r'[<>:"/\\|?*]', "_", title)
    title = re.sub(r"\s+", " ", title).strip().rstrip(". ")

    reserved = {
        "CON", "PRN", "AUX", "NUL",
        *(f"COM{i}" for i in range(1, 10)),
        *(f"LPT{i}" for i in range(1, 10)),
    }

    if title.upper() in reserved:
        title = "_" + title

    return title or "untitled"


def download_track(query: str) -> Path | None:
    """Ищет первое видео и скачивает аудио."""
    print(f"\nПоиск: {query}")

    try:
        results = Search(query).videos

        if not results:
            print("Видео не найдено.")
            return None

        yt = YouTube(results[0].watch_url)

        print(f"Название: {yt.title}")
        print(f"Ссылка: {yt.watch_url}")

        # Предпочитаем WebM/Opus с максимальным битрейтом.
        streams = (
            yt.streams
            .filter(only_audio=True, mime_type="audio/webm")
            .order_by("abr")
            .desc()
        )

        stream = streams.first()

        # Резерв: любой доступный аудиопоток.
        if stream is None:
            stream = yt.streams.get_audio_only()

        if stream is None:
            print("Аудиопоток не найден.")
            return None

        filename = safe_filename(yt.title) + "." + stream.subtype
        destination = OUTPUT_DIR / filename

        if destination.exists() and destination.stat().st_size > 0:
            print(f"Файл уже существует: {destination}")
            return destination

        # Несколько попыток на случай временной ошибки.
        for attempt in range(1, 4):
            try:
                print(
                    f"Скачивание, попытка {attempt}/3, "
                    f"формат: {stream.subtype}, "
                    f"битрейт: {stream.abr}"
                )

                stream.download(
                    output_path=str(OUTPUT_DIR),
                    filename=filename,
                )

                if destination.exists() and destination.stat().st_size > 0:
                    print(f"Скачано: {destination}")
                    return destination

                raise RuntimeError("Файл не создан или пуст.")

            except Exception as exc:
                print(f"Ошибка скачивания: {type(exc).__name__}: {exc}")

                if attempt < 3:
                    time.sleep(attempt * 3)

        return None

    except Exception as exc:
        print(f"Ошибка поиска или получения потока: {type(exc).__name__}: {exc}")
        return None


def separate_stems(audio_path: Path) -> None:
    """Разделяет аудио с помощью Demucs."""
    track_name = safe_filename(audio_path.stem)
    track_stems_dir = STEMS_DIR / track_name

    # Не запускаем модель повторно, если все шесть стемов уже готовы.
    if all(
        (track_stems_dir / stem / f"{stem}.wav").exists()
        for stem in STEM_NAMES
    ):
        print("Все стемы уже существуют, пропускаю.")
        return

    temp_output = TEMP_DIR / track_name
    model_output = temp_output / "htdemucs_6s"

    try:
        print(f"\nРазделение на стемы: {audio_path.name}")

        command = [
            sys.executable,
            "-m", "demucs",
            "-n", "htdemucs_6s",
            "-d", "cpu",
            "-o", str(temp_output),
            str(audio_path),
        ]

        subprocess.run(command, check=True)

        # Demucs обычно создаёт папку по имени исходного файла.
        candidates = list(model_output.glob("*"))

        if not candidates:
            raise FileNotFoundError(
                f"Папка со стемами не найдена: {model_output}"
            )

        demucs_track_dir = candidates[0]
        track_stems_dir.mkdir(parents=True, exist_ok=True)

        found_stems = set()

        for stem_file in demucs_track_dir.glob("*.wav"):
            stem_name = stem_file.stem

            if stem_name not in STEM_NAMES:
                continue

            destination_dir = track_stems_dir / stem_name
            destination_dir.mkdir(parents=True, exist_ok=True)

            destination_file = destination_dir / f"{stem_name}.wav"
            shutil.copy2(stem_file, destination_file)

            found_stems.add(stem_name)
            print(f"Сохранён стем: {destination_file}")

        missing = set(STEM_NAMES) - found_stems

        if missing:
            print(f"Не найдены стемы: {', '.join(sorted(missing))}")
        else:
            print(f"Все стемы сохранены в: {track_stems_dir}")

    except subprocess.CalledProcessError as exc:
        print(f"Ошибка Demucs, код завершения: {exc.returncode}")

    except Exception as exc:
        print(f"Ошибка разделения: {type(exc).__name__}: {exc}")

    finally:
        # Удаляем временные файлы только при наличии всех стемов.
        if all(
            (track_stems_dir / stem / f"{stem}.wav").exists()
            for stem in STEM_NAMES
        ):
            shutil.rmtree(temp_output, ignore_errors=True)


def main() -> None:
    print(f"Аудиофайлы: {OUTPUT_DIR}")
    print(f"Стемы: {STEMS_DIR}")
    print(f"Количество треков: {len(TRACKS)}")

    for index, query in enumerate(TRACKS, start=1):
        print(f"\n{'=' * 60}")
        print(f"[{index}/{len(TRACKS)}] {query}")

        audio_path = download_track(query)

        if audio_path is not None:
            separate_stems(audio_path)
        else:
            print("Трек пропущен: скачать аудио не удалось.")

        time.sleep(2)

    print("\nОбработка завершена.")


if __name__ == "__main__":
    main()


In [ ]:

from pathlib import Path
import re
import shutil
import subprocess
import sys
import time

from pytubefix import Search, YouTube


TRACKS = [
    "Blur - Song 2 (Official Music Video)",
]

OUTPUT_DIR = Path.home() / "Audio Stems" / "YouTube Audio"
STEMS_DIR = OUTPUT_DIR / "Stems"
TEMP_DIR = OUTPUT_DIR / "_demucs_temp"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
STEMS_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

STEM_NAMES = [
    "vocals",
    "drums",
    "bass",
    "guitar",
    "piano",
    "other",
]


def safe_filename(title: str) -> str:
    """Создаёт безопасное имя файла или папки в Windows."""
    title = re.sub(r'[<>:"/\\|?*]', "_", title)
    title = re.sub(r"\s+", " ", title).strip().rstrip(". ")

    reserved = {
        "CON", "PRN", "AUX", "NUL",
        *(f"COM{i}" for i in range(1, 10)),
        *(f"LPT{i}" for i in range(1, 10)),
    }

    if title.upper() in reserved:
        title = "_" + title

    return title or "untitled"


def download_track(query: str) -> Path | None:
    """Ищет первое видео и скачивает аудио."""
    print(f"\nПоиск: {query}")

    try:
        results = Search(query).videos

        if not results:
            print("Видео не найдено.")
            return None

        yt = YouTube(results[0].watch_url)

        print(f"Название: {yt.title}")
        print(f"Ссылка: {yt.watch_url}")

        # Сначала выбираем WebM/Opus с максимальным битрейтом.
        streams = (
            yt.streams
            .filter(only_audio=True, mime_type="audio/webm")
            .order_by("abr")
            .desc()
        )

        stream = streams.first()

        # Резервный вариант: любой доступный аудиопоток.
        if stream is None:
            stream = yt.streams.get_audio_only()

        if stream is None:
            print("Аудиопоток не найден.")
            return None

        filename = safe_filename(yt.title) + "." + stream.subtype
        destination = OUTPUT_DIR / filename

        if destination.exists() and destination.stat().st_size > 0:
            print(f"Файл уже существует: {destination}")
            return destination

        # Повторные попытки при временных ошибках.
        for attempt in range(1, 4):
            try:
                print(
                    f"Скачивание, попытка {attempt}/3; "
                    f"формат: {stream.subtype}; "
                    f"битрейт: {stream.abr}"
                )

                stream.download(
                    output_path=str(OUTPUT_DIR),
                    filename=filename,
                )

                if destination.exists() and destination.stat().st_size > 0:
                    print(f"Скачано: {destination}")
                    return destination

                raise RuntimeError("Файл не создан или пуст.")

            except Exception as exc:
                print(
                    f"Ошибка скачивания: "
                    f"{type(exc).__name__}: {exc}"
                )

                if attempt < 3:
                    time.sleep(attempt * 3)

        return None

    except Exception as exc:
        print(
            f"Ошибка поиска или получения потока: "
            f"{type(exc).__name__}: {exc}"
        )
        return None


def separate_stems(audio_path: Path) -> None:
    """Разделяет аудио на шесть стемов в одной папке."""
    track_name = safe_filename(audio_path.stem)
    track_stems_dir = STEMS_DIR / track_name

    # Пропускаем обработку, если все стемы уже существуют.
    if all(
        (track_stems_dir / f"{stem}.wav").exists()
        for stem in STEM_NAMES
    ):
        print("Все стемы уже существуют, пропускаю.")
        return

    temp_output = TEMP_DIR / track_name
    model_output = temp_output / "htdemucs_6s"

    try:
        print(f"\nРазделение на стемы: {audio_path.name}")

        command = [
            sys.executable,
            "-m", "demucs",
            "-n", "htdemucs_6s",
            "-d", "cpu",
            "-o", str(temp_output),
            str(audio_path),
        ]

        subprocess.run(command, check=True)

        # Находим каталог с результатами Demucs.
        candidates = list(model_output.glob("*"))

        if not candidates:
            raise FileNotFoundError(
                f"Папка со стемами не найдена: {model_output}"
            )

        demucs_track_dir = candidates[0]
        track_stems_dir.mkdir(parents=True, exist_ok=True)

        found_stems = set()

        for stem_file in demucs_track_dir.glob("*.wav"):
            stem_name = stem_file.stem

            if stem_name not in STEM_NAMES:
                continue

            # Все файлы сохраняются непосредственно в папку трека.
            destination_file = track_stems_dir / f"{stem_name}.wav"
            shutil.copy2(stem_file, destination_file)

            found_stems.add(stem_name)
            print(f"Сохранён стем: {destination_file}")

        missing = set(STEM_NAMES) - found_stems

        if missing:
            print(
                "Не найдены стемы: "
                + ", ".join(sorted(missing))
            )
        else:
            print(f"Все стемы сохранены в: {track_stems_dir}")

    except subprocess.CalledProcessError as exc:
        print(f"Ошибка Demucs, код завершения: {exc.returncode}")

    except Exception as exc:
        print(
            f"Ошибка разделения: "
            f"{type(exc).__name__}: {exc}"
        )

    finally:
        # Удаляем временные данные только после получения всех стемов.
        if all(
            (track_stems_dir / f"{stem}.wav").exists()
            for stem in STEM_NAMES
        ):
            shutil.rmtree(temp_output, ignore_errors=True)


def main() -> None:
    print(f"Аудиофайлы: {OUTPUT_DIR}")
    print(f"Стемы: {STEMS_DIR}")
    print(f"Количество треков: {len(TRACKS)}")

    # ЭТАП 1. Скачиваем все треки.
    downloaded_tracks = []

    print("\n" + "=" * 60)
    print("ЭТАП 1: СКАЧИВАНИЕ АУДИО")
    print("=" * 60)

    for index, query in enumerate(TRACKS, start=1):
        print(f"\n[{index}/{len(TRACKS)}] {query}")

        audio_path = download_track(query)

        if audio_path is not None:
            downloaded_tracks.append(audio_path)
        else:
            print("Трек пропущен: скачать аудио не удалось.")

        time.sleep(2)

    print("\n" + "=" * 60)
    print(
        f"Скачивание завершено: "
        f"{len(downloaded_tracks)} из {len(TRACKS)} треков."
    )
    print("=" * 60)

    # ЭТАП 2. Разделяем скачанные треки на стемы.
    print("\n" + "=" * 60)
    print("ЭТАП 2: РАЗДЕЛЕНИЕ НА СТЕМЫ")
    print("=" * 60)

    for index, audio_path in enumerate(downloaded_tracks, start=1):
        print(
            f"\n[{index}/{len(downloaded_tracks)}] "
            f"{audio_path.name}"
        )

        separate_stems(audio_path)

    print("\nОбработка завершена.")


if __name__ == "__main__":
    main()


In [ ]:
pip install -r requirements-test.txt